In [1]:
import pandas as pd
import requests

In [2]:
BASE_URL = "https://api.hearthstonejson.com/v1/latest"

In [3]:
def load_cards(locale: str, collectible_only: bool = False) -> list[dict]:
    file_name = "cards.collectible.json" if collectible_only else "cards.json"
    url = f"{BASE_URL}/{locale}/{file_name}"

    response = requests.get(
        url,
        headers={
            "User-Agent": "Mozilla/5.0 hearthstone-localization-mapper/1.0",
            "Accept": "application/json",
        },
        timeout=60,
    )

    response.raise_for_status()
    return response.json()

In [4]:
def to_name_df(cards: list[dict], lang_suffix: str) -> pd.DataFrame:
    rows = []

    for card in cards:
        rows.append({
            "id": card.get("id"),
            "dbfId": card.get("dbfId"),
            f"name_{lang_suffix}": card.get("name"),
            "type": card.get("type")
        })

    return pd.DataFrame(rows)

In [5]:
en_df = to_name_df(load_cards("enUS"), "en")

In [6]:
pl_df = to_name_df(load_cards("plPL"), "pl")

In [7]:
mapping = en_df.merge(
    pl_df[["id", "name_pl"]],
    on="id",
    how="inner",
)

In [8]:
mapping.loc[:, ["name_en", "name_pl", "type"]].to_excel(
    r"C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\excel-mappingi\hearthstone.xlsx",
    index=False
)